# 04 — Latency Profiling and Benchmarks

Profiles the per-layer latency of the Zero-Trust Deepfake Voice Defense pipeline.

Analyses:
- Preprocessing + feature extraction latency
- CNN inference latency
- Decision engine latency
- Challenge generation latency
- End-to-end pipeline latency
- P50 / P95 / P99 percentiles
- Comparison against latency budgets in `configs/latency_config.yaml`


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.utils.timer import Timer, LatencyTracker
from src.utils.config_loader import load_config

latency_cfg = load_config('../configs/latency_config.yaml')
budgets = latency_cfg.get('budgets', {})
print('Latency budgets (ms):', budgets)


In [ ]:
from src.data.audio_preprocessor import AudioPreprocessor
from src.data.feature_extractor import FeatureExtractor
from src.models.cnn_detector import CNNDetector
from src.decision.threshold_engine import ThresholdEngine
from src.decision.trust_scorer import TrustScorer
from src.liveness.challenge_generator import ChallengeGenerator

N_RUNS = 50
SR = 16_000

# Generate synthetic 2-second test audio
t = np.linspace(0, 2.0, SR * 2, endpoint=False)
test_waveform = np.sin(2 * np.pi * 440 * t).astype(np.float32)

preprocessor = AudioPreprocessor()
extractor = FeatureExtractor()
detector = CNNDetector(backbone='resnet18', pretrained=False, device='cpu').build()
scorer = TrustScorer()
engine = ThresholdEngine()
gen = ChallengeGenerator()

tracker = LatencyTracker()

for _ in range(N_RUNS):
    with Timer() as t_pre:
        proc, _ = preprocessor.process(test_waveform.copy(), SR)
        feat = extractor.extract(proc)
    tracker.record('preprocessing_ms', t_pre.elapsed_ms)

    with Timer() as t_cnn:
        result = detector.predict(feat)
    tracker.record('cnn_inference_ms', t_cnn.elapsed_ms)

    with Timer() as t_dec:
        trust = scorer.score(result['genuine_prob'], 0.75)
        decision = engine.evaluate(trust)
    tracker.record('decision_ms', t_dec.elapsed_ms)

    with Timer() as t_ch:
        _ = gen.generate()
    tracker.record('challenge_generation_ms', t_ch.elapsed_ms)

summary = tracker.summary()
for stage, stats in summary.items():
    budget = budgets.get(stage, None)
    over_budget = '⚠️ OVER BUDGET' if budget and stats['p95_ms'] > budget else '✅'
    print(f'{stage:35s} mean={stats["mean_ms"]:6.1f} ms | p95={stats["p95_ms"]:6.1f} ms {over_budget}')


In [ ]:
# Visualise latency distributions
fig, axes = plt.subplots(1, len(summary), figsize=(4 * len(summary), 4))
for ax, (stage, stats) in zip(axes, summary.items()):
    ax.set_title(stage.replace('_ms', ''), fontsize=9)
    ax.set_xlabel('Latency (ms)')
    ax.axvline(stats['mean_ms'], color='red', linestyle='--', label='mean')
    ax.axvline(stats['p95_ms'], color='orange', linestyle=':', label='p95')
    budget = budgets.get(stage, None)
    if budget:
        ax.axvline(budget, color='green', linestyle='-', label='budget')
    ax.legend(fontsize=7)
plt.suptitle('Latency Distribution by Stage', fontsize=12)
plt.tight_layout()
plt.show()
